# 00 — Run the whole analysis in order

This notebook executes the four analysis notebooks of this kit one after another, in dependency order, and
records how long each one took. It is a convenience: every notebook can also be opened and run on
its own, as long as the notebooks it reads from have run first.

**ReEDS outputs.** No notebook in this kit opens ReEDS output files (`outputs.h5`). Leave the environment
variable `NL_REEDS_OUTPUTS` unset. When it is set, or when a folder `nuclear-learning-reeds-outputs` sits
next to the repository, notebook 01 copies its `mc_perdraw.npz` into the output archive. The regenerated
file differs from the archived one in the last digits, so the copy would break the archive's `SHA256SUMS`.

**Runtime.** The full chain takes about 1.5 hours on an idle machine with 4 threads and up to about
2.5 hours on a loaded one. The long items are 03a (17–50 minutes) and 03b (60–80 minutes). Notebooks 01
and 02 take under 3 minutes each. The table at the end lists the actual times of every notebook that
carries a run stamp, including earlier invocations.

**Skipping finished notebooks.** Set `SKIP_IF_EXECUTED = True` to skip a notebook whose file already
holds executed outputs from a run of this same driver (the driver writes a stamp into the notebook
metadata). The default re-runs everything.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

HERE = Path.cwd().resolve()
assert (HERE / "00_run_all.ipynb").exists(), "run this notebook from z-ethan/notebooks/"

ORDER = [
    "01_monte_carlo_cost_engine",
    "02_atb_basis_and_literature",
    "03a_npv_and_mixed_build",
    "03b_tech_comparison_and_observation",
]
SKIP_IF_EXECUTED = False
STAMP = "nlpaper_run_all"

print("python:", sys.executable)
print("NL_REEDS_OUTPUTS:", os.environ.get("NL_REEDS_OUTPUTS", "(not set, as this kit expects)"))


## Execute

Each notebook is executed in place with `nbconvert`, so the saved file carries the outputs of this
run. Clear them before a commit: the committed notebooks hold no outputs (README, "How to run").
A failure stops the chain and prints the notebook's error; fix the cause and re-run this cell.
Notebooks 02 and 03a read the exports of 01, and 03b reads the exports of 01 and 03a, so the
order above follows the dependencies.


In [ ]:
def already_executed(path):
    try:
        meta = json.loads(path.read_text(encoding="utf-8")).get("metadata", {})
        return STAMP in meta
    except Exception:
        return False

def stamp(path, seconds):
    nb = json.loads(path.read_text(encoding="utf-8"))
    nb.setdefault("metadata", {})[STAMP] = {"seconds": round(seconds, 1), "when": time.strftime("%Y-%m-%d %H:%M:%S")}
    path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

times = []
for name in ORDER:
    path = HERE / f"{name}.ipynb"
    if not path.exists():
        raise FileNotFoundError(path)
    if SKIP_IF_EXECUTED and already_executed(path):
        print(f"skip  {name} (already executed by this driver)")
        continue
    print(f"run   {name} ...", flush=True)
    t0 = time.time()
    cmd = [sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute", "--inplace",
           "--ExecutePreprocessor.kernel_name=python3", "--ExecutePreprocessor.timeout=-1", str(path)]
    res = subprocess.run(cmd, cwd=HERE, capture_output=True, text=True)
    dt = time.time() - t0
    if res.returncode != 0:
        print(res.stdout[-3000:]); print(res.stderr[-6000:])
        raise RuntimeError(f"{name} failed after {dt/60:.1f} min; see the error above")
    stamp(path, dt)
    times.append((name, dt))
    print(f"done  {name}  {dt/60:6.1f} min", flush=True)


## Runtimes of this run

In [ ]:
import pandas as pd
# the stamps written by this or an earlier invocation of the driver, so a resumed run still shows every notebook
rows = []
for name in ORDER:
    meta = json.loads((HERE / f'{name}.ipynb').read_text(encoding='utf-8')).get('metadata', {}).get(STAMP)
    rows.append((name, meta['seconds'] if meta else float('nan'), meta['when'] if meta else ''))
df = pd.DataFrame(rows, columns=['notebook', 'seconds', 'when'])
df['minutes'] = (df['seconds'] / 60).round(1)
print(df.to_string(index=False))
print(f"total: {df['seconds'].sum()/3600:.2f} h")
